# BF-VAE v2 — Beat Enhancement Fine-tuning + Test

## What this notebook does
1. **Setup** — clone repo, install deps, mount Drive  
2. **Fine-tune** — load v1 checkpoint, retrain with *fixed* beat loss (`1 - regularity`)  
3. **Test** — input one weak-beat audio → output strong-beat audio with comparison plots

### Key fixes over v1
| Issue | v1 (wrong) | v2 (fixed) |
|-------|-----------|------------|
| `beat_loss` direction | `= autocorrelation` → trains **weaker** beats | `= 1 - autocorrelation` → trains **stronger** beats |
| MSE coverage | full-spectrum MSE fights beat loss | MSE only on high-freq, low-freq is free for rhythm |
| KL weight | 0.001 (nearly disabled) | 0.01 (gentle regularisation) |

> ▶ **Runtime → Change runtime type → GPU (T4)**

In [ ]:
# ── Cell 1: Environment check ─────────────────────────────────────────────
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU. Go to Runtime → Change runtime type → GPU')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────
!pip install -q librosa soundfile scikit-learn
print('Done.')

In [ ]:
# ── Cell 3: Clone / update repo ───────────────────────────────────────────
import os
REPO   = 'https://github.com/ELina-zhaoCN/VAE-Weak-Beat-Music.git'
PROJ   = '/content/VAE-Weak-Beat-Music'

if not os.path.exists(PROJ):
    !git clone {REPO} {PROJ}
else:
    !cd {PROJ} && git pull

os.chdir(PROJ)
!ls 7.BF_VAE_v2/

In [ ]:
# ── Cell 4: Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE_DIR  = '/content/drive/MyDrive/BF_VAE_v2'
CKPT_V1    = '/content/drive/MyDrive/BF_VAE_results/checkpoints_v2/best_model.pth'
SAVE_DIR   = f'{DRIVE_DIR}/checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

if os.path.exists(CKPT_V1):
    print(f'v1 checkpoint found: {CKPT_V1}  ({os.path.getsize(CKPT_V1)/1e6:.0f} MB)')
else:
    print('v1 checkpoint NOT found at:', CKPT_V1)
    print('Please upload your best_model.pth to Drive/BF_VAE_results/checkpoints_v2/')

In [ ]:
# ── Cell 5: Download & filter FMA small (weak-beat tracks) ───────────────
import glob, shutil
FMA_DIR = '/content/fma_data'
os.makedirs(FMA_DIR, exist_ok=True)

META_ZIP  = f'{FMA_DIR}/fma_metadata.zip'
AUDIO_ZIP = f'{FMA_DIR}/fma_small.zip'

if not os.path.exists(f'{FMA_DIR}/fma_metadata'):
    print('Downloading FMA metadata (~342 MB)...')
    !wget -q --show-progress -c https://os.unil.cloud.switch.ch/fma/fma_metadata.zip -O {META_ZIP}
    !unzip -q {META_ZIP} -d {FMA_DIR}
    print('Metadata OK.')

if not os.path.exists(f'{FMA_DIR}/fma_small'):
    print('Downloading FMA small (~7.2 GB, ~15 min)...')
    !wget -q --show-progress -c https://os.unil.cloud.switch.ch/fma/fma_small.zip -O {AUDIO_ZIP}
    !unzip -q {AUDIO_ZIP} -d {FMA_DIR}
    print('Audio OK.')

# Filter weak-beat genres
import pandas as pd
WEAK_GENRES = ['Ambient','Drone','Noise','Dark Ambient',
               'Field Recording','Experimental','Free Jazz','Avant-Garde','Electroacoustic']

tracks = pd.read_csv(f'{FMA_DIR}/fma_metadata/tracks.csv', index_col=0, header=[0,1])
weak   = tracks[tracks['track']['genre_top'].isin(WEAK_GENRES)]
print(f'Found {len(weak)} weak-beat tracks')

DATA_DIR = '/content/fma_weak_beat'
os.makedirs(DATA_DIR, exist_ok=True)
copied = 0
for tid in weak.index:
    s   = str(tid).zfill(6)
    src = f'{FMA_DIR}/fma_small/{s[:3]}/{s}.mp3'
    dst = f'{DATA_DIR}/{s}.mp3'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        copied += 1

all_files = glob.glob(f'{DATA_DIR}/*.mp3')
print(f'Total training files: {len(all_files)}')

In [ ]:
# ── Cell 6: Quick sanity check — beat_loss v2 direction ───────────────────
import sys
sys.path.insert(0, f'{PROJ}/7.BF_VAE_v2')
sys.path.insert(0, f'{PROJ}/3.Model')
sys.path.insert(0, f'{PROJ}/2.Music_dataset')

from beat_loss_v2 import beat_loss_v2

B, T = 2, 431
# Regular beat: pulses every 22 frames ≈ 120 BPM
regular = torch.zeros(B, 1, 128, T)
for t in range(0, T, 22):
    regular[:, :, :16, t:t+3] = 1.0

ambient = torch.randn(B, 1, 128, T) * 0.05

l_reg = beat_loss_v2(regular)
l_amb = beat_loss_v2(ambient)

print(f'Regular beat loss : {l_reg:.4f}  (should be LOW  ~0.05)')
print(f'Ambient     loss  : {l_amb:.4f}  (should be HIGH ~0.90)')
assert l_reg < l_amb, 'ERROR: loss direction still wrong!'
print('✓ Loss direction confirmed correct')

In [ ]:
# ── Cell 7: Fine-tune training ────────────────────────────────────────────
# Estimated time: ~3-5 min/epoch on T4 with FMA small weak-beat subset
# 30 epochs ≈ 1.5 - 2.5 hours  →  run overnight or use Colab Pro

!python {PROJ}/7.BF_VAE_v2/train_v2.py \
    --data_dir     {DATA_DIR} \
    --checkpoint   {CKPT_V1} \
    --save_dir     {SAVE_DIR} \
    --epochs       40 \
    --batch_size   16 \
    --lr           5e-5 \
    --kl_weight    0.01 \
    --beat_weight  2.0 \
    --warmup_epochs 10 \
    --log_interval  5

In [ ]:
# ── Cell 8: Verify best model was saved ───────────────────────────────────
import json

BEST_CKPT = f'{SAVE_DIR}/best_model_v2.pth'
print(f'Checkpoint exists: {os.path.exists(BEST_CKPT)}')
if os.path.exists(BEST_CKPT):
    print(f'Size: {os.path.getsize(BEST_CKPT)/1e6:.0f} MB')

history_path = f'{SAVE_DIR}/history_v2.json'
if os.path.exists(history_path):
    hist = json.load(open(history_path))
    epochs_done = len(hist)
    best_reg    = max(h['val']['regularity'] for h in hist)
    print(f'Epochs trained: {epochs_done}')
    print(f'Best val regularity: {best_reg:.4f}')
    # Show last 5 epochs
    print('\nLast 5 epochs:')
    print(f'{"Epoch":>6}  {"TrainReg":>9}  {"ValReg":>7}  {"BeatLoss":>9}')
    for h in hist[-5:]:
        print(f'{h["epoch"]:>6}  {h["train"]["regularity"]:>9.4f}  '
              f'{h["val"]["regularity"]:>7.4f}  {h["val"]["beat"]:>9.4f}')

---
## Test: Input Weak Beat → Output Strong Beat

Run the cells below to test the trained model.

**To use your own audio:** upload an `.mp3` or `.wav` file and set `TEST_FILE` below.

In [ ]:
# ── Cell 9: Pick a test file ──────────────────────────────────────────────
import random, glob
random.seed(99)

# Option A: use a random weak-beat file from the dataset
all_test = glob.glob(f'{DATA_DIR}/*.mp3')
TEST_FILE = random.choice(all_test)

# Option B: upload your own file and set path manually
# from google.colab import files
# uploaded = files.upload()
# TEST_FILE = list(uploaded.keys())[0]

print(f'Test file: {TEST_FILE}')

In [ ]:
# ── Cell 10: Run inference & show results ─────────────────────────────────
from inference_v2 import enhance_beats
import sys
sys.path.insert(0, f'{PROJ}/7.BF_VAE_v2')

OUT_WAV  = '/content/output_beat_enhanced.wav'
OUT_PNG  = '/content/comparison.png'

results = enhance_beats(
    input_path      = TEST_FILE,
    checkpoint_path = BEST_CKPT,
    output_path     = OUT_WAV,
    plot_path       = OUT_PNG,
    latent_dim      = 128,
)

from IPython.display import Image
Image(OUT_PNG)

In [ ]:
# ── Cell 11: Listen to original vs enhanced ───────────────────────────────
from IPython.display import Audio, display
import soundfile as sf

SR = 22050
print('▶ Original (weak beat):')
display(Audio(results['audio_in'],  rate=SR))

print('▶ Enhanced (strong beat):')
display(Audio(results['audio_out'], rate=SR))

print(f"\nBeat Regularity: {results['reg_in']:.3f} → {results['reg_out']:.3f} "
      f"({results['reg_delta']:+.3f})")
print(f"BPM: {results['bpm_in']:.1f} → {results['bpm_out']:.1f}")

In [ ]:
# ── Cell 12: Batch test on 5 files ────────────────────────────────────────
import numpy as np
from inference_v2 import enhance_beats

test_files = random.sample(all_test, min(5, len(all_test)))
batch_results = []

print(f'{'File':<28} {'RegIn':>6} {'RegOut':>7} {'Delta':>7} {'BPMIn':>7} {'BPMOut':>7}')
print('-' * 65)

for f in test_files:
    r = enhance_beats(
        input_path      = f,
        checkpoint_path = BEST_CKPT,
        latent_dim      = 128,
    )
    batch_results.append(r)
    name = os.path.basename(f)[:26]
    print(f"{name:<28} {r['reg_in']:>6.3f} {r['reg_out']:>7.3f} "
          f"{r['reg_delta']:>+7.3f} {r['bpm_in']:>7.1f} {r['bpm_out']:>7.1f}")

print('-' * 65)
avg_delta = np.mean([r['reg_delta'] for r in batch_results])
avg_mse   = np.mean([r['mse'] for r in batch_results])
pct_improved = np.mean([r['reg_delta'] > 0 for r in batch_results]) * 100
print(f"\nAvg regularity delta : {avg_delta:+.3f}")
print(f"% tracks improved    : {pct_improved:.0f}%")
print(f"Avg MSE              : {avg_mse:.4f}")

In [ ]:
# ── Cell 13: Save results to Drive ────────────────────────────────────────
RESULTS_DIR = f'{DRIVE_DIR}/test_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

import shutil
shutil.copy(OUT_WAV, f'{RESULTS_DIR}/output_beat_enhanced.wav')
shutil.copy(OUT_PNG, f'{RESULTS_DIR}/comparison.png')

import json
summary = [{
    'file': os.path.basename(r_dict['audio_in'].tobytes()[:8].hex() if hasattr(r_dict['audio_in'], 'tobytes') else str(i)),
    'reg_in':   r_dict['reg_in'],
    'reg_out':  r_dict['reg_out'],
    'reg_delta':r_dict['reg_delta'],
    'bpm_in':   r_dict['bpm_in'],
    'bpm_out':  r_dict['bpm_out'],
    'mse':      r_dict['mse'],
} for i, r_dict in enumerate(batch_results)]

with open(f'{RESULTS_DIR}/batch_results_v2.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Saved to: {RESULTS_DIR}')
print(f'Files: comparison.png, output_beat_enhanced.wav, batch_results_v2.json')